## 1.4 Filtros por Payload: O Diferencial do Qdrant

Esta é a feature que separa o Qdrant de uma busca linear simples com numpy.

Você pode combinar busca vetorial com filtros nos metadados em **uma única operação eficiente**. Por exemplo:
- "Encontre artigos *semanticamente similares à minha query* E que sejam da *categoria tecnologia* E do *ano 2024*"

O Qdrant executa isso de forma otimizada — não é busca vetorial seguida de filtro (que seria lento). O índice HNSW é construído considerando os filtros.

**Estrutura de filtros:**
- `must`: AND — todos os filtros devem ser verdadeiros
- `should`: OR — pelo menos um deve ser verdadeiro
- `must_not`: NOT — nenhum desses deve ser verdadeiro

Isso permite queries complexas: "artigos de tecnologia OU ciência, mas não de política, com score > 0.7".

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue,
    UpdateCollection, HnswConfigDiff,
)
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd

# Conectar ao Qdrant
client = QdrantClient(host='localhost', port=6333)

# Verificar conexao
collections = client.get_collections()
print(f'Qdrant conectado!')
print(f'Collections existentes: {[c.name for c in collections.collections]}')

# Modelo de embedding
model = SentenceTransformer('all-MiniLM-L6-v2')  # 384 dims
print(f'Modelo carregado: 384 dimensoes')

## 2.1 Criar uma Collection

In [ ]:
COLLECTION = 'artigos_tech'

# Recriar collection (limpa se ja existia)
if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)
client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(
        size=384,              # dimensoes do vetor
        distance=Distance.COSINE,  # metrica de distancia
    ),
)

info = client.get_collection(COLLECTION)
print(f'Collection criada: {COLLECTION}')
print(f'  Status: {info.status}')
print(f'  Dimensoes: {info.config.params.vectors.size}')
print(f'  Distancia: {info.config.params.vectors.distance}')
print(f'\nAcesse: http://localhost:6333/dashboard para ver no browser')

## 1.2 Inserindo Documentos (Upsert)

Cada documento no Qdrant é chamado de **Point** e tem três componentes:
- **`id`**: identificador único (int ou UUID)
- **`vector`**: o embedding do documento (lista de floats)
- **`payload`**: metadados em JSON livre — qualquer coisa que você queira armazenar e filtrar depois

O **payload** é uma das features mais poderosas do Qdrant. Diferente de bancos vetoriais simples, você pode armazenar informações ricas junto com cada vetor e depois filtrar por elas sem sair do banco.

`upsert` (update + insert): se o ID já existir, atualiza. Se não existir, cria. Isso facilita reindexação incremental.

In [ ]:
## 1.2 Inserindo Documentos (Upsert)

Cada documento no Qdrant é chamado de **Point** e tem três componentes:
- **`id`**: identificador único (int ou UUID)
- **`vector`**: o embedding do documento (lista de floats)
- **`payload`**: metadados em JSON livre — qualquer coisa que você queira armazenar e filtrar depois

O **payload** é uma das features mais poderosas do Qdrant. Diferente de bancos vetoriais simples, você pode armazenar informações ricas junto com cada vetor e depois filtrar por elas sem sair do banco.

`upsert` (update + insert): se o ID já existir, atualiza. Se não existir, cria. Isso facilita reindexação incremental.

## 2.3 Busca Semantica

In [ ]:
def buscar(query, limit=3, filtro_categoria=None):
    query_vec = model.encode(query, normalize_embeddings=True)
    
    query_filter = None
    if filtro_categoria:
        query_filter = Filter(
            must=[FieldCondition(
                key='categoria',
                match=MatchValue(value=filtro_categoria)
            )]
        )
    
    results = client.query_points(
        collection_name=COLLECTION,
        query=query_vec.tolist(),
        limit=limit,
        query_filter=query_filter,
        with_payload=True,
    ).points
    return results

# Teste 1: busca sem filtro
print('BUSCA: Como armazenar vetores eficientemente?')
print('-' * 50)
for r in buscar('Como armazenar vetores eficientemente?', limit=3):
    print(f'Score: {r.score:.3f} | [{r.payload["categoria"]}] {r.payload["texto"]}')

print()

# Teste 2: busca com filtro de categoria
print('BUSCA (filtro: categoria=rag): Como construir pipeline?')
print('-' * 50)
for r in buscar('Como construir pipeline de perguntas e respostas?', limit=3, filtro_categoria='rag'):
    print(f'Score: {r.score:.3f} | [{r.payload["categoria"]}] {r.payload["texto"]}')

## 2.4 Operacoes CRUD

In [ ]:
# READ: buscar um ponto por ID
pontos = client.retrieve(
    collection_name=COLLECTION,
    ids=[1, 3],
    with_payload=True,
    with_vectors=False,
)
print('Pontos recuperados por ID:')
for p in pontos:
    print(f'  ID {p.id}: {p.payload["texto"][:60]}')

# UPDATE: atualizar payload de um ponto
client.set_payload(
    collection_name=COLLECTION,
    payload={'atualizado': True, 'versao': 2},
    points=[1],
)
print('\nPayload atualizado no ponto 1')

# Verificar
p = client.retrieve(COLLECTION, ids=[1], with_payload=True)[0]
print(f'Payload do ponto 1: {p.payload}')

# DELETE: deletar um ponto
# client.delete(COLLECTION, points_selector=[ponto_id])  # descomentado = deleta!
print('\nDelete: demonstrado mas nao executado para preservar os dados')

## 1.4 Filtros por Payload: O Diferencial do Qdrant

Esta é a feature que separa o Qdrant de uma busca linear simples com numpy.

Você pode combinar busca vetorial com filtros nos metadados em **uma única operação eficiente**. Por exemplo:
- "Encontre artigos *semanticamente similares à minha query* E que sejam da *categoria tecnologia* E do *ano 2024*"

O Qdrant executa isso de forma otimizada — não é busca vetorial seguida de filtro (que seria lento). O índice HNSW é construído considerando os filtros.

**Estrutura de filtros:**
- `must`: AND — todos os filtros devem ser verdadeiros
- `should`: OR — pelo menos um deve ser verdadeiro
- `must_not`: NOT — nenhum desses deve ser verdadeiro

Isso permite queries complexas: "artigos de tecnologia OU ciência, mas não de política, com score > 0.7".

In [ ]:
from qdrant_client.models import (
    Filter, FieldCondition, MatchValue, Range
)

query_vec = model.encode('indexacao vetorial', normalize_embeddings=True)

# Filtro por intervalo de ano
resultado_filtrado = client.query_points(
    collection_name=COLLECTION,
    query=query_vec.tolist(),
    limit=5,
    query_filter=Filter(
        must=[
            FieldCondition(
                key='ano',
                range=Range(gte=2022)  # ano >= 2022
            )
        ]
    ),
    with_payload=True,
).points

print('Busca: "indexacao vetorial" (filtro: ano >= 2022):')
for r in resultado_filtrado:
    print(f'  Score: {r.score:.3f} | Ano: {r.payload["ano"]} | {r.payload["texto"][:60]}')

print()

# Filtro NOT (excluir categoria)
from qdrant_client.models import Filter as QFilter
resultado_sem_rag = client.query_points(
    collection_name=COLLECTION,
    query=query_vec.tolist(),
    limit=5,
    query_filter=Filter(
        must_not=[
            FieldCondition(key='categoria', match=MatchValue(value='rag'))
        ]
    ),
    with_payload=True,
).points

print('Busca: "indexacao vetorial" (excluindo categoria=rag):')
for r in resultado_sem_rag:
    print(f'  Score: {r.score:.3f} | [{r.payload["categoria"]}] {r.payload["texto"][:60]}')

### Por que filtros no banco vetorial são tão importantes para RAG?

Imagine um sistema RAG para uma empresa com documentos de vários departamentos. Quando um funcionário do RH faz uma pergunta, você não quer que o sistema retorne documentos confidenciais de finanças — mesmo que sejam semanticamente similares.

Com filtros de payload, você pode garantir: "busque apenas documentos onde `department == 'HR'` ou `visibility == 'public'`". Isso é essencial para sistemas multi-tenant ou com controle de acesso.

Outra aplicação comum: filtrar por data. "Mostre apenas documentos dos últimos 6 meses" — você adiciona `created_at` ao payload e filtra na query.

## 2.6 Scroll (paginar todos os pontos)

In [ ]:
# Listar todos os pontos sem query (scroll)
all_points, next_cursor = client.scroll(
    collection_name=COLLECTION,
    limit=10,
    with_payload=True,
    with_vectors=False,
)

print(f'Total de pontos na collection: {len(all_points)}')
print('\nTodos os documentos indexados:')
for p in all_points:
    print(f'  ID {p.id} | [{p.payload["categoria"]}] {p.payload["texto"][:60]}')

## 1.2 Inserindo Documentos (Upsert)

Cada documento no Qdrant é chamado de **Point** e tem três componentes:
- **`id`**: identificador único (int ou UUID)
- **`vector`**: o embedding do documento (lista de floats)
- **`payload`**: metadados em JSON livre — qualquer coisa que você queira armazenar e filtrar depois

O **payload** é uma das features mais poderosas do Qdrant. Diferente de bancos vetoriais simples, você pode armazenar informações ricas junto com cada vetor e depois filtrar por elas sem sair do banco.

`upsert` (update + insert): se o ID já existir, atualiza. Se não existir, cria. Isso facilita reindexação incremental.

## Resumo

| Operação | Método Qdrant | Uso |
|----------|--------------|-----|
| Criar índice | `create_collection` | Uma vez, na inicialização |
| Indexar docs | `upsert` | Indexação inicial e incremental |
| Buscar | `search` | Cada query do usuário |
| Buscar + filtrar | `search` + `query_filter` | RAG multi-tenant, filtros temporais |
| Ler por ID | `retrieve` | Debug, auditoria |
| Deletar | `delete` | Remover docs desatualizados |
| Atualizar metadata | `set_payload` | Sem precisar re-embedar |

**O que vem a seguir:**
- [02 — HNSW Indexing](02_hnsw_indexing.html): como o Qdrant faz buscas tão rápidas?
- [03 — Quantization](03_quantization.html): como comprimir o índice para economizar memória